# Community Detection on Multilayer Networks

This notebook demonstrates how to detect and analyze communities in multilayer networks.

## What is Community Detection?

Community detection identifies groups of nodes that are more densely connected to each other than to the rest of the network. In multilayer networks, communities can:

* Exist within single layers
* Span multiple layers
* Emerge from inter-layer coupling

## Installation

In [ ]:
!pip install py3plex -q

## Setup: Create a Multilayer Network

We'll create a network with clear community structure.

In [ ]:
from py3plex.core import multinet

# Create network
network = multinet.multi_layer_network(directed=False)

# Group 1: Alice, Bob, Charlie (well connected)
# Group 2: David, Eve, Frank (well connected)
# Sparse connections between groups

edges = [
    # Group 1 - Social layer
    ['Alice', 'social', 'Bob', 'social', 1],
    ['Bob', 'social', 'Charlie', 'social', 1],
    ['Alice', 'social', 'Charlie', 'social', 1],
    
    # Group 2 - Social layer
    ['David', 'social', 'Eve', 'social', 1],
    ['Eve', 'social', 'Frank', 'social', 1],
    ['David', 'social', 'Frank', 'social', 1],
    
    # Weak inter-group connection
    ['Charlie', 'social', 'David', 'social', 1],
    
    # Group 1 - Work layer
    ['Alice', 'work', 'Bob', 'work', 1],
    ['Bob', 'work', 'Charlie', 'work', 1],
    
    # Group 2 - Work layer
    ['David', 'work', 'Eve', 'work', 1],
    ['Eve', 'work', 'Frank', 'work', 1],
]

network.add_edges(edges, input_type="list")

print("Network created:")
network.basic_stats()

## Method 1: Louvain Algorithm

The Louvain algorithm is fast and optimizes modularity.

In [ ]:
from py3plex.algorithms.community_detection.multilayer_modularity import louvain_multilayer
from collections import Counter

# Run Louvain community detection
partition_dict, modularity = louvain_multilayer(network)

print(f"Modularity: {modularity:.3f}")
print(f"Number of communities: {len(set(partition_dict.values()))}")

# Count community sizes
comm_sizes = Counter(partition_dict.values())
print(f"\nCommunity sizes: {dict(comm_sizes)}")

# Show assignments
print("\nCommunity assignments:")
for node, community in sorted(partition_dict.items()):
    print(f"  {node} → Community {community}")

## Analyze Community Structure

Let's examine the communities we found.

In [ ]:
# Group nodes by community
communities = {}
for node, comm_id in partition_dict.items():
    if comm_id not in communities:
        communities[comm_id] = []
    communities[comm_id].append(node)

# Display communities
for comm_id, nodes in sorted(communities.items()):
    print(f"\nCommunity {comm_id}:")
    print(f"  Size: {len(nodes)} nodes")
    print(f"  Members: {nodes[:10]}")  # Show first 10

## Use DSL to Analyze Communities

Query nodes by community using the DSL.

In [ ]:
from py3plex.dsl import Q, L

# Store partition in network
network.assign_partition(partition_dict)

# Query nodes in community 0
result = (
    Q.nodes()
     .where(community=0)
     .compute("degree")
     .execute(network)
)

df = result.to_pandas()
print("Nodes in community 0:")
print(df)

## Visualize Communities

Create a visualization with nodes colored by community.

In [ ]:
from py3plex.visualization.multilayer import draw_multilayer_default
import matplotlib.pyplot as plt

# Create color map based on communities
colors = {}
color_palette = ['red', 'blue', 'green', 'orange', 'purple', 'brown']

for node, comm_id in partition_dict.items():
    colors[node] = color_palette[comm_id % len(color_palette)]

# Visualize
plt.figure(figsize=(12, 8))
draw_multilayer_default(
    [network],
    display=True,
    node_colors=colors
)
plt.title("Multilayer Network with Detected Communities")
plt.show()

## Compare Single-Layer vs Multilayer Detection

See how multilayer detection differs from single-layer analysis.

In [ ]:
from py3plex.algorithms.community_detection.community_wrapper import louvain_communities

# Multilayer detection (we already have this)
print("Multilayer Communities:")
ml_communities = len(set(partition_dict.values()))
print(f"  Number of communities: {ml_communities}")
print(f"  Modularity: {modularity:.3f}")

# Single-layer detection (on flattened network)
single_partition = louvain_communities(network)
sl_communities = len(set(single_partition.values()))

print("\nSingle-Layer Communities (flattened):")
print(f"  Number of communities: {sl_communities}")

print("\nMultilayer detection can reveal layer-specific structure!")

## Layer-Specific Communities

Analyze community structure within each layer separately.

In [ ]:
# Get nodes by layer and community
for layer in ['social', 'work']:
    print(f"\n{layer.upper()} LAYER:")
    layer_nodes = [node for node in partition_dict.keys() if node[1] == layer]
    layer_communities = {partition_dict[node] for node in layer_nodes}
    
    print(f"  Nodes: {len(layer_nodes)}")
    print(f"  Communities: {len(layer_communities)}")
    
    for comm_id in sorted(layer_communities):
        members = [node[0] for node in layer_nodes if partition_dict[node] == comm_id]
        print(f"    Community {comm_id}: {members}")

## Compute Community Metrics

Calculate statistics about the detected communities.

In [ ]:
import numpy as np

# Community size statistics
sizes = list(comm_sizes.values())

print("Community Size Statistics:")
print(f"  Mean: {np.mean(sizes):.2f}")
print(f"  Median: {np.median(sizes):.2f}")
print(f"  Min: {np.min(sizes)}")
print(f"  Max: {np.max(sizes)}")
print(f"  Std Dev: {np.std(sizes):.2f}")

# Plot size distribution
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.bar(range(len(sizes)), sorted(sizes, reverse=True))
plt.xlabel('Community Rank')
plt.ylabel('Community Size')
plt.title('Community Size Distribution')
plt.grid(True, alpha=0.3)
plt.show()

## Null Model Comparison

Compare detected communities against null models for statistical significance.

In [ ]:
from py3plex.nullmodels import configuration_model
from py3plex.algorithms.community_detection.multilayer_modularity import louvain_multilayer

# Statistical significance testing using null models
# Null models randomize the network while preserving key properties

# Step 1: Detect communities in original network
partition_orig, Q_orig = louvain_multilayer(network)
n_communities_orig = len(set(partition_orig.values()))

print(f"Original network: Q={Q_orig:.3f}, Communities={n_communities_orig}")

# Step 2: Generate configuration model (randomized network)
# Preserves degree distribution but randomizes connections
null_network = configuration_model(network, n_swaps=100)

# Step 3: Detect communities in null model
partition_null, Q_null = louvain_multilayer(null_network)
n_communities_null = len(set(partition_null.values()))

print(f"Null model:       Q={Q_null:.3f}, Communities={n_communities_null}")
print(f"\nModularity difference: {Q_orig - Q_null:.3f}")

# Interpretation: If original Q >> null Q, communities are significant
if Q_orig > Q_null + 0.05:  # Threshold can vary
    print("✓ Original network has significantly stronger community structure!")
else:
    print("✗ Community structure may not be significant.")

## Community Detection with Uncertainty

Quantify uncertainty in community assignments using bootstrap.

In [ ]:
from py3plex.uncertainty import estimate_uncertainty
import numpy as np

# Quantify uncertainty in community detection using bootstrap
# Bootstrap resamples edges to create variation
n_bootstrap = 20
modularity_values = []
community_counts = []

for i in range(n_bootstrap):
    # Create bootstrap sample by resampling edges with replacement
    edges = list(network.get_edges(data=True))
    bootstrap_edges = [edges[i] for i in np.random.choice(len(edges), len(edges), replace=True)]
    
    # Build bootstrap network
    from py3plex.core import multinet
    bootstrap_net = multinet.multi_layer_network()
    for u, v, data in bootstrap_edges:
        bootstrap_net.add_edge(u, v, **data)
    
    # Run community detection on bootstrap sample
    partition, Q = louvain_multilayer(bootstrap_net)
    modularity_values.append(Q)
    community_counts.append(len(set(partition.values())))

# Compute statistics: mean, std, confidence intervals
Q_mean = np.mean(modularity_values)
Q_std = np.std(modularity_values)
Q_ci_low = np.percentile(modularity_values, 2.5)  # 95% CI lower bound
Q_ci_high = np.percentile(modularity_values, 97.5)  # 95% CI upper bound

print(f"\nModularity (over {n_bootstrap} bootstrap samples):")
print(f"  Mean: {Q_mean:.3f}")
print(f"  Std:  {Q_std:.3f}")
print(f"  95% CI: [{Q_ci_low:.3f}, {Q_ci_high:.3f}]")

comm_mean = np.mean(community_counts)
comm_std = np.std(community_counts)
print(f"\nNumber of communities:")
print(f"  Mean: {comm_mean:.1f}")
print(f"  Std:  {comm_std:.1f}")
print(f"  Range: [{min(community_counts)}, {max(community_counts)}]")

## Plugin System for Custom Algorithms

Extend py3plex with custom community detection plugins.

In [ ]:
from py3plex.plugins import CommunityPlugin, PluginRegistry

# Plugin system allows extending py3plex with custom algorithms
# Define a custom community detection plugin
class SimpleCommunityPlugin(CommunityPlugin):
    """Example plugin: assigns random communities (for demonstration)."""
    
    @property
    def name(self) -> str:
        return "random_community"
    
    @property
    def description(self) -> str:
        return "Assigns random community labels (demonstration only)"
    
    def compute(self, network, **kwargs):
        """Compute community assignments."""
        import random
        nodes = list(network.get_nodes())
        n_communities = kwargs.get('n_communities', 3)
        # Return dict mapping node -> community_id
        return {node: random.randint(0, n_communities - 1) for node in nodes}

# Register and use the plugin
registry = PluginRegistry()
registry.register(SimpleCommunityPlugin())

# Use the plugin through the registry
plugin = registry.get_plugin('random_community')
print(f"\nPlugin: {plugin.name}")
print(f"Description: {plugin.description}")

partition = plugin.compute(network, n_communities=3)
print(f"\nAssigned {len(set(partition.values()))} communities to {len(partition)} nodes")
print(f"Sample assignments: {dict(list(partition.items())[:5])}")

# List all available plugins
print(f"\nRegistered plugins: {registry.list_plugins()}")

## Summary

In this tutorial, you learned:

* ✅ Louvain algorithm for community detection
* ✅ Multilayer vs single-layer comparison
* ✅ Community visualization techniques
* ✅ DSL queries for community analysis
* ✅ Layer-specific communities
* ✅ Community metrics computation
* ✅ Null model comparison for significance testing
* ✅ Uncertainty quantification with bootstrap
* ✅ Plugin system for custom algorithms

## Next Steps

* Try [other community detection algorithms](https://skblaz.github.io/py3plex/reference/community.html)
* Learn about [dynamics simulation](simulate_dynamics.ipynb)
* Explore the [DSL tutorial](query_with_dsl.ipynb)
* Read about [plugin development](https://skblaz.github.io/py3plex/guides/plugins.html)